# MatchMiner-AI on Mayo notes (local Gemma 4)

Minimal patient -> trial matching run on a Mayo note file, using the local
llama.cpp Gemma 4 server (`config_local_llamacpp.yaml`, see `README_local_run.md`).

**Real patient data.** Inputs are read from, and all results are written to,
`H:\data\mm-ai\` - never this repo. Cells print only counts, so the notebook
itself holds no patient content; open the files in `OUTPUT_DIR` to review
results. Do not add cells that display summaries/IDs, and clear outputs before
committing anything.

Start the LLM server first: `H:\Apps\llama.cpp\run_server_gemma4e2b.bat`.
Run with the working directory set to `examples/`.

In [ ]:
import logging
import os
import time
from pathlib import Path

import pandas as pd

from matchminer_ai import load_config

# Real patient data: read from and write to locations outside the repo.
NOTES_PATH = Path(r"H:\data\mm-ai\mayo_dataset_10.csv")
OUTPUT_DIR = Path(r"H:\data\mm-ai\output_mayo_local")

# Trial side: reuse the trial spaces already extracted by run_examples.ipynb
# with the same local Gemma 4 model (10 example trials).
TRIAL_SUMMARIES_PATH = Path("output_local_llamacpp/trial_summaries.csv")
TRIALS_PATH = Path("data/trials_for_summarize.csv")

repo_root = Path.cwd().resolve().parent
assert (repo_root / "pyproject.toml").exists(), "run this notebook from examples/"
assert repo_root not in OUTPUT_DIR.resolve().parents, "OUTPUT_DIR must be outside the repo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Progress goes to the cell output and to OUTPUT_DIR/run.log (counts and row
# indices only, no note text). Follow a headless run with:
#   Get-Content -Wait H:\data\mm-ai\output_mayo_local\run.log
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(), logging.FileHandler(OUTPUT_DIR / "run.log", mode="w")],
    force=True,
)
log = logging.getLogger("mayo_run")
t0 = time.time()


def step(n, name):
    log.info("===== [step %d/5] %s (elapsed %.1f min) =====", n, name, (time.time() - t0) / 60)


os.environ.setdefault("OPENAI_API_KEY", "not-needed")
config = load_config("config_local_llamacpp.yaml")

## 1. Load and clean notes

In [ ]:
step(1, "Load and clean notes")

raw_notes = pd.read_csv(NOTES_PATH, dtype=str)

notes = raw_notes[["patient_id", "event_type", "note_text", "note_date"]].copy()
notes["note_date"] = pd.to_datetime(notes["note_date"], errors="coerce")

# Drop rows broken by unescaped quotes/newlines in the export (text spilled
# into patient_id / extra "Unnamed" columns) and rows without a usable date.
valid = (
    notes["patient_id"].str.fullmatch(r"\d+", na=False)
    & notes["note_text"].notna()
    & notes["note_date"].notna()
    & raw_notes.filter(like="Unnamed").isna().all(axis=1)
)
notes = notes[valid].reset_index(drop=True)

print(f"kept {len(notes)} / {len(raw_notes)} rows ({(~valid).sum()} dropped)")
print(f"patients: {notes['patient_id'].nunique()}")
print(notes["event_type"].value_counts().to_string())

## 2. Summarize patients (Gemma 4 via llama.cpp)

In [ ]:
from matchminer_ai.patients import summarize_patients

# Runs in rounds: round k sends chunk k of every patient that has one.
# Each "HTTP Request ... 200 OK" line is one finished chunk; each
# "Progress: x/y completed" line is one finished round.
step(2, "Summarize patients with Gemma 4 (slowest step)")
patient_summaries, patient_metadata, patient_qc = summarize_patients(
    notes, config=config, return_metadata=True, return_qc=True
)
patient_summaries.to_csv(OUTPUT_DIR / "patient_summaries.csv", index=None)
patient_qc.to_csv(OUTPUT_DIR / "patient_qc_report.csv", index=None)

print(f"patient summaries: {len(patient_summaries)}")
patient_qc[["metric", "value", "denominator", "percent"]]

## 3. Load trial spaces

In [ ]:
step(3, "Load trial spaces")

trial_results = pd.read_csv(TRIAL_SUMMARIES_PATH)
trials = pd.read_csv(TRIALS_PATH)
print(f"trial spaces: {len(trial_results)} across {trial_results['trial_id'].nunique()} trials")

## 4. Embed, retrieve candidates, score

In [ ]:
from matchminer_ai.embedding import embed_for_matching
from matchminer_ai.matching import (
    exclusion_criteria_check,
    generate_candidate_matches,
    score_match_quality,
)

step(4, "Embed, retrieve candidates, run TrialChecker + BoilerplateChecker")
embedded_trials = embed_for_matching(trial_results, entity_type="trial", config=config)
embedded_patients = embed_for_matching(patient_summaries, entity_type="patient", config=config)

candidate_matches = generate_candidate_matches(
    query_df=embedded_patients, corpus_df=embedded_trials
)

# TrialChecker: patient summary vs. each candidate trial space
match_quality_matches = score_match_quality(
    candidate_matches.merge(
        patient_summaries[["patient_id", "cancer_history_summary"]], on="patient_id"
    ).merge(
        trial_results[["space_trial_id", "clinical_space_summary"]], on="space_trial_id"
    ),
    config=config,
)
match_quality_matches["trial_id"] = match_quality_matches["space_trial_id"].str.split("-").str[0]

# BoilerplateChecker: one check per patient/trial pair
exclusion_results = exclusion_criteria_check(
    match_quality_matches[["patient_id", "trial_id"]].drop_duplicates().merge(
        patient_summaries[["patient_id", "general_exclusion_criteria_evidence"]], on="patient_id"
    ).merge(
        trial_results[["trial_id", "general_exclusion_criteria"]].drop_duplicates(), on="trial_id"
    ),
    config=config,
)

print(f"candidate pairs: {len(candidate_matches)}")
print(exclusion_results["exclusion_criteria_pass"].value_counts().to_string())

## 5. Save final matches

One row per patient/trial pair (its top-scoring clinical space), ranked by
match-quality score within each patient.

In [ ]:
step(5, "Save final matches")

final_matches = (
    match_quality_matches.merge(
        candidate_matches[["patient_id", "space_trial_id", "similarity_score"]],
        on=["patient_id", "space_trial_id"],
    )
    .merge(exclusion_results, on=["patient_id", "trial_id"], how="left")
    .merge(
        trial_results[["space_trial_id", "clinical_space_number", "general_exclusion_criteria"]],
        on="space_trial_id",
    )
    .merge(trials[["trial_id", "trial_title"]].drop_duplicates("trial_id"), on="trial_id", how="left")
    .sort_values("match_quality_score", ascending=False)
    .drop_duplicates(["patient_id", "trial_id"])
    .sort_values(["patient_id", "match_quality_score"], ascending=[True, False])
    .reset_index(drop=True)
)
final_matches.insert(2, "match_rank", final_matches.groupby("patient_id").cumcount() + 1)
final_matches.to_csv(OUTPUT_DIR / "final_matches.csv", index=None)

log.info("done in %.1f min", (time.time() - t0) / 60)
print(f"final matches: {len(final_matches)} rows -> {OUTPUT_DIR / 'final_matches.csv'}")
final_matches["match_quality_score"].describe()